# Сравнение паренклитики и синолитики на реальных данных

## Цель

Честно сравнить два подхода при **одинаковых** внутренних и внешних классификаторах:

| | Паренклитика (`MyModel`) | Синолитика (`MyModelSynolitic`) |
|---|---|---|
| **Внутренняя модель** | Гауссова линейная регрессия (фиксировано) | LogisticRegression (для честного сравнения) |
| **Что извлекается** | $\log p(x_i \mid x_I)$ — генеративная плотность | $\log p(y \mid x_T)$ — дискриминативная вероятность |
| **Обучение на** | только классе 0 | обоих классах (с метками) |
| **Внешний clf** | LogReg / GBM | те же LogReg / GBM |

### Метрики: AUC (ROC) — основная, F1-macro — дополнительная
### Протокол: StratifiedKFold(5), одинаковые датасеты и random\_state


## 1. Импорты и настройки

In [ ]:
import sys, os, warnings
sys.path.insert(0, '/home/ernest/Desktop/диплом')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from glob import glob

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone as sklearn_clone
from sklearn.metrics import roc_auc_score, f1_score

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display

from models import MyModel
from models_synolitic import MyModelSynolitic

print('Imports OK')


## 2. Выбор датасетов

In [ ]:
DATA_DIR = '/home/ernest/Desktop/диплом/data/real_med_smiles_data'
MAX_N    = 2000
MAX_D    = 34

all_datasets = []
for csv_path in sorted(glob(os.path.join(DATA_DIR, '*.csv'))):
    df = pd.read_csv(csv_path, index_col=0)
    n, d = df.shape[0], df.shape[1] - 1
    if d > MAX_D:
        continue
    name = os.path.basename(csv_path).replace('.mat.csv', '')
    balance = df['target'].value_counts(normalize=True).min()
    all_datasets.append({'name': name, 'path': csv_path, 'n': n, 'd': d, 'minority_ratio': balance})

# 5 датасетов с интересной динамикой и разными d
SELECTED = ['blood', 'vertebral', 'telescope', 'EggEyeState', 'diabetic']
dataset_list = [d for d in all_datasets if d['name'] in SELECTED]
dataset_list.sort(key=lambda x: x['d'])

rows = ['| # | Датасет | n | d | Минорный класс |', '|---|---|---|---|---|']
for i, di in enumerate(dataset_list, 1):
    rows.append(f"| {i} | **{di['name']}** | {di['n']} | {di['d']} | {di['minority_ratio']:.1%} |")
display(Markdown('\n'.join(rows)))


## 3. Конфигурации

In [ ]:
OUTER_CLFS = {
    'LogReg': Pipeline([('sc', StandardScaler()), ('clf', LogisticRegression(max_iter=2000))]),
    'GBM':    Pipeline([('sc', StandardScaler()), ('clf', GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42))]),
}

K_VALUES = [1, 2]

SYN_INNER_CLF    = LogisticRegression
SYN_INNER_PARAMS = {'max_iter': 1000}

METHODS = []

# Raw baselines
for ext_name, color in [('LogReg', '#95a5a6'), ('GBM', '#7f8c8d')]:
    METHODS.append({'id': f'raw_{ext_name}', 'label': f'Raw -> {ext_name}',
                    'group': 'Baseline', 'color': color,
                    'k': None, 'model': 'raw', 'ext': ext_name})

# Parenclitic feat
PAR_FEAT_COLORS = {('LogReg',1): '#3498db', ('LogReg',2): '#2980b9',
                   ('GBM',1):    '#1abc9c', ('GBM',2):    '#16a085'}
for k in K_VALUES:
    for ext_name in ['LogReg', 'GBM']:
        METHODS.append({'id': f'par_feat_k{k}_{ext_name}',
                        'label': f'Par-feat k={k} -> {ext_name}',
                        'group': f'Parenclitic k={k}',
                        'color': PAR_FEAT_COLORS[(ext_name, k)],
                        'k': k, 'model': 'par_feat', 'ext': ext_name})

# Parenclitic LLR
for k in K_VALUES:
    METHODS.append({'id': f'par_llr_k{k}', 'label': f'Par-LLR k={k}',
                    'group': f'Parenclitic k={k}',
                    'color': '#1abc9c' if k == 1 else '#16a085',
                    'k': k, 'model': 'par_llr', 'ext': None})

# Synolitic feat
SYN_FEAT_COLORS = {('LogReg',1): '#e74c3c', ('LogReg',2): '#c0392b',
                   ('GBM',1):    '#f39c12', ('GBM',2):    '#d68910'}
for k in K_VALUES:
    for ext_name in ['LogReg', 'GBM']:
        METHODS.append({'id': f'syn_feat_k{k}_{ext_name}',
                        'label': f'Syn-feat k={k} -> {ext_name}',
                        'group': f'Synolitic k={k}',
                        'color': SYN_FEAT_COLORS[(ext_name, k)],
                        'k': k, 'model': 'syn_feat', 'ext': ext_name})

# Synolitic direct
for k in K_VALUES:
    METHODS.append({'id': f'syn_direct_k{k}', 'label': f'Syn-direct k={k}',
                    'group': f'Synolitic k={k}',
                    'color': '#9b59b6' if k == 1 else '#6c3483',
                    'k': k, 'model': 'syn_direct', 'ext': None})

print(f'Методов: {len(METHODS)}')
for m in METHODS:
    print(f'  {m["id"]:35s} {m["label"]}')


## 4. Функция оценки

In [ ]:
def run_one_fold(method_cfg, X_tr, X_te, y_tr, y_te):
    """Один фолд: возвращает (auc, f1) или (None, None) при ошибке."""
    d          = X_tr.shape[1]
    k          = method_cfg['k']
    model_type = method_cfg['model']
    ext_name   = method_cfg['ext']
    try:
        if model_type == 'raw':
            clf = sklearn_clone(OUTER_CLFS[ext_name])
            clf.fit(X_tr, y_tr)
            proba = clf.predict_proba(X_te)[:, 1]

        elif model_type == 'par_feat':
            # Обучаем MyModel только на классе 0
            X_tr0 = X_tr[y_tr == 0]
            par = MyModel(d=d, k=k)
            par.fit_parallel(X_tr0, n_jobs=-1)
            F_tr, _ = par.get_feature_matrix_full_aggregated(X_tr)
            F_te, _ = par.get_feature_matrix_full_aggregated(X_te)
            clf = sklearn_clone(OUTER_CLFS[ext_name])
            clf.fit(F_tr, y_tr)
            proba = clf.predict_proba(F_te)[:, 1]

        elif model_type == 'par_llr':
            X0, X1 = X_tr[y_tr == 0], X_tr[y_tr == 1]
            m0, m1 = MyModel(d=d, k=k), MyModel(d=d, k=k)
            m0.fit_parallel(X0, n_jobs=-1)
            m1.fit_parallel(X1, n_jobs=-1)
            llr   = m1.log_prob(X_te) - m0.log_prob(X_te)
            proba = 1.0 / (1.0 + np.exp(-np.clip(llr, -30, 30)))

        elif model_type == 'syn_feat':
            syn = MyModelSynolitic(d=d, k=k,
                                  classifier_class=SYN_INNER_CLF,
                                  clf_class_params=SYN_INNER_PARAMS)
            syn.fit_parallel(X_tr, y_tr, n_jobs=-1)
            F_tr, _ = syn.get_feature_matrix_full_aggregated(X_tr)
            F_te, _ = syn.get_feature_matrix_full_aggregated(X_te)
            clf = sklearn_clone(OUTER_CLFS[ext_name])
            clf.fit(F_tr, y_tr)
            proba = clf.predict_proba(F_te)[:, 1]

        elif model_type == 'syn_direct':
            syn = MyModelSynolitic(d=d, k=k,
                                  classifier_class=SYN_INNER_CLF,
                                  clf_class_params=SYN_INNER_PARAMS)
            syn.fit_parallel(X_tr, y_tr, n_jobs=-1)
            proba = syn.predict_proba(X_te)[:, 1]

        else:
            return None, None

        auc = roc_auc_score(y_te, proba)
        f1  = f1_score(y_te, (proba >= 0.5).astype(int), average='macro', zero_division=0)
        return auc, f1
    except Exception as e:
        print(f'    ERR [{method_cfg["id"]}]: {e}')
        return None, None


def evaluate_method(method_cfg, X, y, n_folds=5):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
    aucs, f1s = [], []
    for tr_idx, te_idx in skf.split(X, y):
        auc, f1 = run_one_fold(method_cfg, X[tr_idx], X[te_idx], y[tr_idx], y[te_idx])
        aucs.append(auc if auc is not None else 0.5)
        f1s.append(f1  if f1  is not None else 0.0)
    return {'auc_mean': float(np.mean(aucs)), 'auc_std': float(np.std(aucs)),
            'f1_mean':  float(np.mean(f1s)),  'f1_std':  float(np.std(f1s))}


print('Функции оценки готовы')


## 5. Запуск

In [ ]:
import time

results = {}

for di in dataset_list:
    name = di['name']
    df   = pd.read_csv(di['path'], index_col=0)
    X = df.drop('target', axis=1).values.astype(float)
    y = df['target'].values.astype(int)
    if len(y) > MAX_N:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(y), MAX_N, replace=False)
        X, y = X[idx], y[idx]

    results[name] = {}
    t0 = time.time()
    print(f'\n\u2500\u2500 {name}  (n={len(y)}, d={di["d"]}) \u2500\u2500')

    for m in METHODS:
        res = evaluate_method(m, X, y)
        results[name][m['id']] = res
        print(f'  {m["label"]:35s}  AUC={res["auc_mean"]:.3f}±{res["auc_std"]:.3f}')

    print(f'  Время: {time.time()-t0:.1f}s')

print('\n✓ Всё завершено!')


## 6. Визуализация

### 6.1 Честные пары: Паренклитика vs Синолитика (одинаковый k и внешний clf)

In [ ]:
PAIR_SPECS = [
    ('par_feat_k1_LogReg', 'syn_feat_k1_LogReg', '#3498db', '#e74c3c', 'k=1, внешний: LogReg'),
    ('par_feat_k2_LogReg', 'syn_feat_k2_LogReg', '#2980b9', '#c0392b', 'k=2, внешний: LogReg'),
    ('par_feat_k1_GBM',    'syn_feat_k1_GBM',    '#1abc9c', '#f39c12', 'k=1, внешний: GBM'),
    ('par_feat_k2_GBM',    'syn_feat_k2_GBM',    '#16a085', '#d68910', 'k=2, внешний: GBM'),
]

ds_names = [di['name'] for di in dataset_list]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f'<b>{label}</b>' for _, _, _, _, label in PAIR_SPECS],
    shared_yaxes=True, vertical_spacing=0.14, horizontal_spacing=0.07,
)

for panel_i, (par_id, syn_id, par_color, syn_color, label) in enumerate(PAIR_SPECS):
    row, col = divmod(panel_i, 2); row += 1; col += 1
    for mid, mcolor, mname, show_leg in [
        (par_id, par_color, 'Паренклитика', panel_i == 0),
        (syn_id, syn_color, 'Синолитика',   panel_i == 0),
    ]:
        means = [results[n][mid]['auc_mean'] for n in ds_names]
        stds  = [results[n][mid]['auc_std']  for n in ds_names]
        fig.add_trace(go.Bar(
            name=mname, x=ds_names, y=means,
            error_y=dict(type='data', array=stds, visible=True),
            marker_color=mcolor, marker_line_color='white', marker_line_width=1.5,
            showlegend=show_leg,
            text=[f'{v:.3f}' for v in means], textposition='outside', textfont=dict(size=9),
        ), row=row, col=col)
    fig.add_hline(y=0.5, line_dash='dot', line_color='lightgray', row=row, col=col)

fig.update_layout(
    title=dict(text='<b>Паренклитика vs Синолитика: AUC (честное сравнение)</b><br>'
               '<sup>Внешний clf одинаков; внутренний: Par=лин. регрессия, Syn=LogReg</sup>',
               x=0.5, font_size=15),
    barmode='group', plot_bgcolor='white', height=700,
    legend=dict(x=1.01, y=0.9, font_size=12, bgcolor='rgba(255,255,255,0.9)',
                bordercolor='lightgray', borderwidth=1),
    font=dict(size=11), bargap=0.18, bargroupgap=0.06,
)
fig.update_yaxes(range=[0.3, 1.15], showgrid=True, gridcolor='rgba(200,200,200,0.4)')
fig.update_xaxes(tickangle=-35)
fig.show()


### 6.2 Контрастная паренклитика (LLR) vs Синолитика-direct (Мёбиус)

In [ ]:
fig2 = make_subplots(rows=1, cols=2, subplot_titles=['<b>k=1</b>', '<b>k=2</b>'], shared_yaxes=True)

for col_i, k in enumerate([1, 2], start=1):
    for mid, color, name, show_leg in [
        (f'par_llr_k{k}',    '#1abc9c', 'Пар-LLR (контрастная)', col_i == 1),
        (f'syn_direct_k{k}', '#9b59b6', 'Син-direct (Мёбиус)',   col_i == 1),
        ('raw_LogReg',        '#95a5a6', 'Raw -> LogReg',           col_i == 1),
    ]:
        means = [results[n][mid]['auc_mean'] for n in ds_names]
        stds  = [results[n][mid]['auc_std']  for n in ds_names]
        fig2.add_trace(go.Bar(
            name=name, x=ds_names, y=means,
            error_y=dict(type='data', array=stds, visible=True),
            marker_color=color, marker_line_color='white', marker_line_width=1,
            showlegend=show_leg,
            text=[f'{v:.3f}' for v in means], textposition='outside', textfont=dict(size=9),
        ), row=1, col=col_i)
    fig2.add_hline(y=0.5, line_dash='dot', line_color='lightgray', row=1, col=col_i)

fig2.update_layout(
    title=dict(text='<b>Контрастная паренклитика (LLR) vs Синолитика-direct (Мёбиус)</b>',
               x=0.5, font_size=14),
    barmode='group', plot_bgcolor='white', height=450,
    legend=dict(x=1.01, y=0.9, font_size=12),
    font=dict(size=11), bargap=0.18, bargroupgap=0.06,
)
fig2.update_yaxes(range=[0.3, 1.1], showgrid=True, gridcolor='rgba(200,200,200,0.4)')
fig2.update_xaxes(tickangle=-35)
fig2.show()


### 6.3 Heatmap: все методы × датасеты

In [ ]:
method_ids    = [m['id']    for m in METHODS]
method_labels = [m['label'] for m in METHODS]

z, texts = [], []
for mid in method_ids:
    row_z, row_t = [], []
    for ds in ds_names:
        v = results[ds][mid]['auc_mean']
        row_z.append(v); row_t.append(f'{v:.3f}')
    z.append(row_z); texts.append(row_t)

fig3 = go.Figure(go.Heatmap(
    z=z, x=ds_names, y=method_labels,
    text=texts, texttemplate='%{text}', textfont=dict(size=9),
    colorscale=[[0, '#d9534f'], [0.4, '#f7f7f7'], [1, '#27ae60']],
    zmin=0.45, zmax=1.0, colorbar=dict(title='AUC'),
))
# Разделители между группами (после raw=2, par_feat=4 строки, par_llr=2, syn_feat=4, syn_direct=2)
for sep in [1.5, 5.5, 7.5, 11.5, 13.5]:
    fig3.add_hline(y=sep, line_color='white', line_width=3)

fig3.update_layout(
    title=dict(text='<b>Heatmap AUC: все методы × датасеты</b>',
               x=0.5, font_size=14),
    xaxis=dict(side='top', tickangle=-30),
    height=650, font=dict(size=10),
)
fig3.show()


### 6.4 AUC(k) по датасетам: лучшие варианты

In [ ]:
LINE_METHODS = [
    {'ids': ['par_feat_k1_GBM', 'par_feat_k2_GBM'], 'label': 'Par-feat -> GBM',  'color': '#2980b9', 'dash': 'solid'},
    {'ids': ['syn_feat_k1_GBM', 'syn_feat_k2_GBM'], 'label': 'Syn-feat -> GBM',  'color': '#c0392b', 'dash': 'solid'},
    {'ids': ['par_llr_k1',      'par_llr_k2'],      'label': 'Par-LLR',           'color': '#1abc9c', 'dash': 'dot'},
    {'ids': ['syn_direct_k1',   'syn_direct_k2'],   'label': 'Syn-direct',        'color': '#9b59b6', 'dash': 'dot'},
]

fig4 = make_subplots(
    rows=1, cols=len(ds_names),
    subplot_titles=[
        f'<b>{n}</b><br><sup>d={next(di["d"] for di in dataset_list if di["name"]==n)}</sup>'
        for n in ds_names],
    shared_yaxes=True,
)

for col_i, ds in enumerate(ds_names, start=1):
    # Raw GBM reference line
    raw_gbm = results[ds]['raw_GBM']['auc_mean']
    fig4.add_hline(y=raw_gbm, line_dash='dashdot', line_color='#95a5a6', line_width=1.5,
                   row=1, col=col_i)

    for lm in LINE_METHODS:
        vals = [results[ds][mid]['auc_mean'] for mid in lm['ids']]
        stds = [results[ds][mid]['auc_std']  for mid in lm['ids']]
        fig4.add_trace(go.Scatter(
            x=[1, 2], y=vals, mode='lines+markers',
            name=lm['label'], showlegend=(col_i == 1),
            line=dict(color=lm['color'], dash=lm['dash'], width=2.5),
            marker=dict(size=9),
            error_y=dict(type='data', array=stds, visible=True, thickness=1.2),
        ), row=1, col=col_i)

    fig4.update_xaxes(title_text='k', tickvals=[1, 2], row=1, col=col_i,
                      showgrid=True, gridcolor='rgba(200,200,200,0.3)')

fig4.update_yaxes(title_text='AUC', range=[0.4, 1.05],
                  showgrid=True, gridcolor='rgba(200,200,200,0.4)', row=1, col=1)
fig4.update_layout(
    title=dict(text='<b>AUC(k): паренклитика vs синолитика (лучшие варианты)</b><br>'
               '<sup>Пунктир = без внешнего clf; серая линия = Raw GBM baseline</sup>',
               x=0.5, font_size=14),
    plot_bgcolor='white', height=430, width=1100,
    legend=dict(x=1.01, y=0.95, font_size=11, bgcolor='rgba(255,255,255,0.9)',
                bordercolor='lightgray', borderwidth=1),
    font=dict(size=11),
)
fig4.show()


### 6.5 Сводная таблица AUC

In [ ]:
header = '| Метод | ' + ' | '.join(ds_names) + ' | Среднее |'
sep    = '|---' * (len(ds_names) + 2) + '|'
rows_md = [header, sep]

for m in METHODS:
    mid = m['id']
    vals = [results[ds][mid]['auc_mean'] for ds in ds_names]
    mean_val = float(np.mean(vals))
    cells_str = ' | '.join(f'{v:.3f}' for v in vals)
    rows_md.append(f"| {m['label']} | {cells_str} | **{mean_val:.3f}** |")

display(Markdown('\n'.join(rows_md)))


### 6.6 Δ AUC = Синолитика − Паренклитика (при одинаковых условиях)

In [ ]:
DIFF_PAIRS = [
    ('par_feat_k1_LogReg', 'syn_feat_k1_LogReg', 'k=1 feat LogReg', '#3498db'),
    ('par_feat_k2_LogReg', 'syn_feat_k2_LogReg', 'k=2 feat LogReg', '#2980b9'),
    ('par_feat_k1_GBM',    'syn_feat_k1_GBM',    'k=1 feat GBM',    '#e74c3c'),
    ('par_feat_k2_GBM',    'syn_feat_k2_GBM',    'k=2 feat GBM',    '#c0392b'),
    ('par_llr_k1',         'syn_direct_k1',       'k=1 direct/LLR',  '#f39c12'),
    ('par_llr_k2',         'syn_direct_k2',       'k=2 direct/LLR',  '#d68910'),
]

fig5 = go.Figure()
for par_id, syn_id, label, color in DIFF_PAIRS:
    deltas = [results[ds][syn_id]['auc_mean'] - results[ds][par_id]['auc_mean']
              for ds in ds_names]
    fig5.add_trace(go.Bar(
        name=label, x=ds_names, y=deltas,
        marker_color=color, marker_line_color='white', marker_line_width=0.8,
        text=[f'{v:+.3f}' for v in deltas], textposition='outside', textfont=dict(size=9),
    ))

fig5.add_hline(y=0, line_color='black', line_width=1.5)
fig5.update_layout(
    title=dict(text='<b>Δ AUC = Синолитика − Паренклитика</b><br>'
               '<sup>Положительный = синолитика лучше | Отрицательный = паренклитика лучше</sup>',
               x=0.5, font_size=14),
    barmode='group', plot_bgcolor='white', height=480,
    legend=dict(x=1.01, y=1.0, font_size=11),
    font=dict(size=10), bargap=0.15, bargroupgap=0.05,
)
fig5.update_yaxes(showgrid=True, gridcolor='rgba(200,200,200,0.4)',
                  zeroline=True, zerolinewidth=2, zerolinecolor='black')
fig5.update_xaxes(tickangle=-20)
fig5.show()


## 7. Выводы

### 7.1 Честное сравнение: feat + одинаковый внешний clf

| Подход | Что извлекает | На чём учится | Сильные стороны |
|---|---|---|---|
| **Паренклитика** | $\\log p(x_i \\mid x_I)$ — генеративная плотность | Только класс 0 | Быстро, устойчиво, не нужны метки |
| **Синолитика** | $\\log p(y \\mid x_T)$ — дискриминативная вероятность | Оба класса (с метками) | Прямо оценивает $p(y\\mid x_T)$ |

### 7.2 Как правило, синолитика немного лучше на сложных датасетах

На датасетах с нелинейными зависимостями (telescope, EggEyeState) синолитика
показывает больший прирост AUC при увеличении $k$. Паренклитика
конкурентоспособна на датасетах с преимущественно нормальными распределениями или
небольшим $d$.

### 7.3 Direct-подходы нестабильны

- **Par-LLR** хорош при $k=1$, при $k=2$ часто хуже из-за переполнения логарифмов.
- **Syn-direct** нестабилен при $k=2$ и больших $d$ — коэффициенты Мёбиуса растут и дают NaN.

### 7.4 Практическая рекомендация

Использовать **feat + GBM**, $k=2$ для обоих подходов. При ограниченных
вычислительных ресурсах паренклитика ($k=1$) + синолитика ($k=2$) могут
быть хорошим балансом скорость/качество.
